# TRELLIS.2 + FLUX.2 notebook quick start

1. Select the Jupyter kernel for the phase2_3d Conda environment and confirm the project root is ~/OEC.
2. Run cells from top to bottom. TRELLIS runs in this kernel; FLUX runs in the separate flux-api model service so a stale or moving Diffusers install cannot break the final cell.
3. Setup cells must pass OEC StorageManager admission before installing TRELLIS dependencies, compiling extensions, or downloading TRELLIS weights. Never bypass a denied preflight.
4. TRELLIS runs on port 7862. The FLUX Gradio client runs on port 7863 and calls the FLUX model service at 127.0.0.1:7867. Keep FLUX service inference on a different idle A100 from any currently loaded TRELLIS model.

Before using FLUX, start its server from the OEC repo root and choose an idle A100 that is not occupied by this kernel's TRELLIS run:
```bash
python tools/oec_service_manager.py start flux-api --gpu <ID> --port 7867 --allow-download
python tools/oec_service_manager.py status flux-api
```
Use --allow-download only when the cache is absent and StorageManager admits the write. The API is loopback-only and the Gradio share link is disabled.

### SSH tunnels

Print the compute node name from the notebook with !hostname. Replace <COMPUTE_NODE> with that exact host name:
```bash
ssh -N -L 5402:<COMPUTE_NODE>:8888 rahkus@login.npsf.cdac.in
ssh -N -L 7862:<COMPUTE_NODE>:7862 rahkus@login.npsf.cdac.in
ssh -N -L 7863:<COMPUTE_NODE>:7863 rahkus@login.npsf.cdac.in
```
Keep these tunnels in separate terminals while using Jupyter and Gradio.

In [ ]:
# 1. THE FROZEN RULE: Core Config First
# Keep shared-cache purging disabled unless an operator has verified no concurrent loaders.
ENABLE_DISK_PURGE = False
import core_config
print("[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.")
from core_config import SYSTEM_DIRS

# 2. ISOLATE GPU: Lock this entire notebook to a specific GPU to prevent VRAM collisions
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Change to 0, 1, 2, or 3
print(f"[SYSTEM] Notebook strictly locked to GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")


In [ ]:
# Replaced the old hard-coded "kill 9 <PID>" command with a read-only process listing.
# Inspect the owner/command and stop only a verified stale process manually.
!ps -u "$USER" -o pid,etime,cmd | grep -E '[g]radio|[p]ipeline_trellis' || true

In [ ]:
!nvidia-smi

In [ ]:
# Verify notebook UI dependencies without upgrading packages during model setup.
import importlib.metadata

for package in ("jupyter", "ipywidgets", "ipykernel"):
    try:
        print(f"{package}=={importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}=MISSING (install only in the intended phase2_3d environment after storage preflight)")


# Step 1: TRELLIS.2 (Model-server client)
TRELLIS uses the managed \`trellis-api\` sidecar. No native extension builds, pip installs, or in-kernel model loads are performed by this notebook.

Start the backend from the OEC repository root on an admitted idle A100:
\`\`\`bash
python tools/oec_service_manager.py start trellis-api --gpu <ID> --port 7868 --allow-download
python tools/oec_service_manager.py status trellis-api
\`\`\`

In [ ]:
import json
import os
import urllib.parse
import requests

TRELLIS_API_BASE = os.environ.get("OEC_TRELLIS_API_URL", "http://127.0.0.1:7868").rstrip("/")
if urllib.parse.urlsplit(TRELLIS_API_BASE).hostname not in {"127.0.0.1", "localhost", "::1"}:
    raise RuntimeError("TRELLIS notebook only permits a loopback model-server endpoint.")
TRELLIS_HTTP = requests.Session()
TRELLIS_HTTP.trust_env = False

def trellis_service_health():
    try:
        response = TRELLIS_HTTP.get(f"{TRELLIS_API_BASE}/health", timeout=8)
        payload = response.json()
    except Exception as exc:
        raise gr.Error(
            f"TRELLIS model server is unreachable at {TRELLIS_API_BASE}. "
            "Start trellis-api on an admitted idle A100 and inspect service logs."
        ) from exc
    if response.status_code != 200 or not payload.get("ready") or not payload.get("model_loaded"):
        raise gr.Error(f"TRELLIS model server is not ready: {json.dumps(payload, ensure_ascii=False)}")
    return payload

# Step 2: FLUX.2 (Model-server client)
The FLUX studio below is a Gradio frontend only. It does not import Diffusers, load weights into this notebook kernel, or install/change the phase2_3d dependency stack. It calls the separate flux-api service on 127.0.0.1:7867. If the TRELLIS setup cell changed package files, restart the kernel before continuing. If the TRELLIS pipeline is resident on this kernel's GPU, run FLUX on another idle A100.

In [ ]:
import importlib.metadata as metadata

# The FLUX studio is a client. Do not pip-install/upgrade Diffusers from this notebook:
# the model server owns its pinned Diffusers/Hugging Face stack in phase2_3d.
for package in ("gradio", "requests", "Pillow"):
    try:
        print(f"{package}=={metadata.version(package)}")
    except metadata.PackageNotFoundError as exc:
        raise RuntimeError(
            f"FLUX client dependency {package!r} is missing from {__import__('sys').executable}. "
            "Use the configured phase2_3d kernel; do not install or upgrade model packages in this notebook."
        ) from exc

import gradio as gr
import requests
from PIL import Image
print("FLUX_SERVER_CLIENT_DEPENDENCIES=PASS")

# Step 3: TRELLIS.2 (Interactive Studio - Image Upload)

In [ ]:
import base64
import hashlib
import json
import os
import struct
import time
from pathlib import Path

import gradio as gr
from PIL import Image

TRELLIS_RESULT_ROOT = Path("/tmp") / "oec-notebook-results"
TRELLIS_RESULT_ROOT.mkdir(parents=True, exist_ok=True)

def validate_glb_file(path: Path):
    data = path.read_bytes()
    if len(data) < 1024 or data[:4] != b"glTF":
        raise gr.Error("TRELLIS returned a missing, empty, or non-GLB asset.")
    version, declared_length = struct.unpack_from("<II", data, 4)
    if version != 2 or declared_length != len(data):
        raise gr.Error(f"Invalid GLB header: version={version}, declared={declared_length}, actual={len(data)}.")
    json_length, chunk_type = struct.unpack_from("<II", data, 12)
    if chunk_type != 0x4E4F534A or json_length <= 0 or 20 + json_length > len(data):
        raise gr.Error("Invalid GLB: first chunk is not a complete JSON chunk.")
    try:
        document = json.loads(data[20:20 + json_length].rstrip(b" \t\r\n\0").decode("utf-8"))
    except Exception as exc:
        raise gr.Error("TRELLIS GLB JSON chunk cannot be decoded.") from exc
    if not isinstance(document, dict) or "asset" not in document:
        raise gr.Error("TRELLIS GLB JSON chunk lacks the required glTF asset object.")
    return {
        "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest(),
        "glb_version": version, "declared_length": declared_length,
        "json_chunk_bytes": json_length, "mesh_count": len(document.get("meshes", [])),
        "node_count": len(document.get("nodes", [])),
    }

def generate_3d(image_path, prompt, max_image_size, texture_size, decimation_target, remesh):
    if not image_path:
        raise gr.Error("Upload an input image.")
    path = Path(str(image_path)).expanduser().resolve()
    if not path.is_file() or path.is_symlink():
        raise gr.Error("Input image path is missing or unsafe.")
    health = trellis_service_health()
    raw_image = path.read_bytes()
    if not raw_image or len(raw_image) > 12 * 1024 * 1024:
        raise gr.Error("Input image must be non-empty and no larger than 12 MiB.")
    try:
        with Image.open(path) as probe:
            if probe.format not in {"PNG", "JPEG", "WEBP"}:
                raise ValueError(f"unsupported image format: {probe.format}")
            probe.verify()
    except Exception as exc:
        raise gr.Error("Input must be a valid PNG, JPEG or WEBP image.") from exc
    payload = {
        "task": "image_to_3d",
        "prompt": (prompt or "Generate a clean, detailed 3D asset").strip()[:800],
        "filename": path.name,
        "parameters": {
            "max_image_size": int(max_image_size), "texture_size": int(texture_size),
            "decimation_target": int(decimation_target), "remesh": bool(remesh),
        },
        "image_base64": base64.b64encode(raw_image).decode("ascii"),
    }
    response = TRELLIS_HTTP.post(f"{TRELLIS_API_BASE}/v1/jobs", json=payload, timeout=40)
    if response.status_code != 202:
        raise gr.Error(f"TRELLIS job rejected (HTTP {response.status_code}): {response.text[:1200]}")
    accepted = response.json()
    job_id = accepted.get("job_id")
    if not isinstance(job_id, str) or not job_id.startswith("job_"):
        raise gr.Error(f"Unexpected TRELLIS job response: {accepted}")
    deadline = time.monotonic() + 1800
    latest = accepted
    while time.monotonic() < deadline:
        poll = TRELLIS_HTTP.get(f"{TRELLIS_API_BASE}/v1/jobs/{job_id}", timeout=20)
        if poll.status_code != 200:
            raise gr.Error(f"TRELLIS job status lookup failed (HTTP {poll.status_code}): {poll.text[:800]}")
        latest = poll.json()
        status = latest.get("status")
        if status == "succeeded":
            break
        if status in {"failed", "error", "cancelled"}:
            raise gr.Error(f"TRELLIS job {status}: {latest.get('message', 'No error details returned')}")
        time.sleep(2)
    else:
        raise gr.Error("TRELLIS inference timed out; inspect trellis-api service logs.")
    result = TRELLIS_HTTP.get(f"{TRELLIS_API_BASE}/v1/jobs/{job_id}/result", timeout=180)
    if result.status_code != 200:
        raise gr.Error(f"TRELLIS asset retrieval failed (HTTP {result.status_code}): {result.text[:800]}")
    content_type = result.headers.get("Content-Type", "").split(";", 1)[0].lower()
    if content_type not in {"model/gltf-binary", "application/octet-stream", "model/glb"}:
        raise gr.Error(f"Expected a binary GLB, received Content-Type={content_type!r}.")
    output_path = TRELLIS_RESULT_ROOT / f"{job_id}.glb"
    if output_path.exists():
        raise gr.Error("Refusing to overwrite an existing notebook result.")
    output_path.write_bytes(result.content)
    try:
        validation = validate_glb_file(output_path)
    except Exception:
        output_path.unlink(missing_ok=True)
        raise
    details = {
        "job_id": job_id, "status": latest.get("status"), "api": TRELLIS_API_BASE,
        "health": health, "metadata": latest.get("metadata", {}),
        "artifact_path": str(output_path), "validation": validation,
        "input_filename": path.name, "license_boundary": "TRELLIS.2 terms must be reviewed for intended use",
    }
    return str(output_path), json.dumps(details, indent=2, ensure_ascii=False)

with gr.Blocks(title="TRELLIS.2 — managed model server") as demo_trellis:
    gr.Markdown("# TRELLIS.2 — image to 3D")
    gr.Markdown("Inference and model weights are managed by the separate trellis-api process. The notebook downloads and validates the GLB returned by the API.")
    with gr.Row():
        with gr.Column():
            input_image = gr.Image(label="Input image", type="filepath")
            prompt = gr.Textbox(label="Prompt metadata (not model-conditioned)", value="A clean, detailed 3D asset", lines=2)
            max_image_size = gr.Dropdown([512, 768, 1024, 1280], value=512, label="Maximum input image size")
            texture_size = gr.Dropdown([512, 1024, 2048, 4096], value=512, label="Texture size")
            decimation_target = gr.Slider(1000, 500000, value=100000, step=1000, label="Decimation target")
            remesh = gr.Checkbox(value=False, label="Remesh")
            generate_button = gr.Button("Generate 3D model", variant="primary")
        with gr.Column():
            output_model = gr.Model3D(label="Validated GLB")
            output_metadata = gr.Textbox(label="Job and GLB metadata", lines=14)
    generate_button.click(
        fn=generate_3d,
        inputs=[input_image, prompt, max_image_size, texture_size, decimation_target, remesh],
        outputs=[output_model, output_metadata],
    )

if os.environ.get("OEC_GRADIO_HEADLESS_SMOKE") != "1":
    demo_trellis.queue(max_size=1).launch(
        server_name="127.0.0.1", server_port=7862, inline=False, share=False, show_error=True, allowed_paths=[str(TRELLIS_RESULT_ROOT)]
    )

# Step 4: FLUX.2 (Interactive Studio - Text to Image)

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import random
import time
import urllib.parse

import gradio as gr
import requests
from PIL import Image

# UI and backend have separate roles: Diffusers/PyTorch/model weights live only in flux-api.
FLUX_API_BASE = os.environ.get("OEC_FLUX_API_URL", "http://127.0.0.1:7867").rstrip("/")
FLUX_RESULT_ROOT = Path("/tmp") / "oec-notebook-results"
FLUX_RESULT_ROOT.mkdir(parents=True, exist_ok=True)
if urllib.parse.urlsplit(FLUX_API_BASE).hostname not in {"127.0.0.1", "localhost", "::1"}:
    raise RuntimeError("FLUX notebook only permits a loopback model-server endpoint.")
FLUX_HTTP = requests.Session()
FLUX_HTTP.trust_env = False


def _flux_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def flux_service_health():
    try:
        response = FLUX_HTTP.get(f"{FLUX_API_BASE}/health", timeout=8)
        payload = response.json()
    except Exception as exc:
        raise gr.Error(
            f"FLUX model server is unreachable at {FLUX_API_BASE}. "
            "Start flux-api on an idle A100 and inspect its service-manager logs."
        ) from exc
    if response.status_code != 200 or not payload.get("ready") or not payload.get("model_loaded"):
        raise gr.Error(f"FLUX model server is not ready: {json.dumps(payload, ensure_ascii=False)}")
    return payload


def generate_concept(prompt: str, seed: float, randomize: bool):
    if not isinstance(prompt, str) or not prompt.strip() or len(prompt) > 800:
        raise gr.Error("Enter an image prompt containing 1–800 characters.")
    seed = random.randint(0, 2**31 - 1) if randomize else int(seed)
    if not 0 <= seed <= 2**31 - 1:
        raise gr.Error("Seed must be between 0 and 2147483647.")
    health = flux_service_health()
    payload = {
        "task": "text_to_image",
        "prompt": prompt.strip(),
        "parameters": {"seed": seed, "width": 512, "height": 512, "steps": 4, "guidance_scale": 1.0},
    }
    try:
        response = FLUX_HTTP.post(f"{FLUX_API_BASE}/v1/jobs", json=payload, timeout=30)
    except Exception as exc:
        raise gr.Error(f"FLUX job submission failed: {type(exc).__name__}") from exc
    if response.status_code != 202:
        raise gr.Error(f"FLUX job submission rejected (HTTP {response.status_code}): {response.text[:1000]}")
    accepted = response.json()
    job_id = accepted.get("job_id")
    if not isinstance(job_id, str) or not job_id.startswith("job_"):
        raise gr.Error(f"Unexpected FLUX job response: {accepted}")

    deadline = time.monotonic() + 1800
    latest = accepted
    while time.monotonic() < deadline:
        poll = FLUX_HTTP.get(f"{FLUX_API_BASE}/v1/jobs/{job_id}", timeout=20)
        if poll.status_code != 200:
            raise gr.Error(f"FLUX job status failed (HTTP {poll.status_code}): {poll.text[:800]}")
        latest = poll.json()
        status = latest.get("status")
        if status == "succeeded":
            break
        if status in {"failed", "error", "cancelled"}:
            raise gr.Error(f"FLUX job {status}: {latest.get('message', 'No error details returned')}")
        time.sleep(2)
    else:
        raise gr.Error("FLUX image generation timed out; inspect flux-api service logs.")

    result = FLUX_HTTP.get(f"{FLUX_API_BASE}/v1/jobs/{job_id}/result", timeout=120)
    if result.status_code != 200:
        raise gr.Error(f"FLUX result download failed (HTTP {result.status_code}): {result.text[:800]}")
    content_type = result.headers.get("Content-Type", "").split(";", 1)[0].lower()
    if content_type != "image/png":
        raise gr.Error(f"Expected image/png from FLUX, received {content_type!r}.")
    out_path = FLUX_RESULT_ROOT / f"{job_id}.png"
    if out_path.exists():
        raise gr.Error("Refusing to overwrite an existing FLUX notebook result.")
    out_path.write_bytes(result.content)
    if out_path.stat().st_size <= 64 or out_path.read_bytes()[:8] != b"\x89PNG\r\n\x1a\n":
        out_path.unlink(missing_ok=True)
        raise gr.Error("FLUX returned an empty or invalid PNG.")
    try:
        with Image.open(out_path) as check:
            check.verify()
        with Image.open(out_path) as check:
            width, height = check.size
            if width <= 0 or height <= 0:
                raise ValueError("empty dimensions")
    except Exception as exc:
        out_path.unlink(missing_ok=True)
        raise gr.Error("FLUX result has a PNG signature but cannot be decoded as an image.") from exc
    details = {
        "job_id": job_id, "status": latest.get("status"), "api": FLUX_API_BASE,
        "health": health, "metadata": latest.get("metadata", {}),
        "artifact_path": str(out_path), "artifact_bytes": out_path.stat().st_size,
        "artifact_sha256": _flux_sha256(out_path), "width": width, "height": height, "seed": seed,
    }
    return str(out_path), seed, json.dumps(details, indent=2, ensure_ascii=False)


with gr.Blocks(title="FLUX.2 Klein — model-server studio") as demo_flux:
    gr.Markdown("# FLUX.2 Klein — text to image")
    gr.Markdown("The API owns model loading and writes the canonical PNG under OEC-managed outputs. This notebook validates and previews the returned PNG.")
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="Prompt", value="A photorealistic amber glass sphere on a clean studio background", lines=4)
            seed = gr.Number(label="Seed", value=42, precision=0)
            randomize = gr.Checkbox(label="Randomize seed", value=True)
            generate_btn = gr.Button("Generate", variant="primary")
        with gr.Column(scale=2):
            output_image = gr.Image(type="filepath", label="FLUX.2 PNG")
            seed_used = gr.Number(label="Seed used", precision=0)
            output_metadata = gr.Textbox(label="Server and artifact metadata", lines=12)
    generate_btn.click(fn=generate_concept, inputs=[prompt, seed, randomize],
                       outputs=[output_image, seed_used, output_metadata])

if os.environ.get("OEC_GRADIO_HEADLESS_SMOKE") != "1":
    print("Starting FLUX Gradio client on port 7863 (share=False; backend=127.0.0.1:7867).")
    demo_flux.queue(max_size=1).launch(server_name="127.0.0.1", server_port=7863,
                                      inline=False, share=False, show_error=True, allowed_paths=[str(FLUX_RESULT_ROOT)])

In [ ]:
# demo.close()

In [ ]:
!nvidia-smi